# Foundations — ETL Walkthrough

Runs `etl_pipeline.py`'s actual functions (imported, not duplicated) one step at a time, so each transformation and each data-quality fix can be shown with real before/after output — easier to demo than reading the `.py` file top to bottom.

Covers Task 1.1 (`projects.csv`) and Task 1.3 (`employees.csv`).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("."))  # so `import etl_pipeline` finds the sibling .py file

import pandas as pd
from etl_pipeline import (
    load_projects, transform_projects,
    load_employees, clean_employees,
    DATA_DIR,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

## Task 1.1 — `projects.csv`

### Step 1 — Load the raw file

The `INFO` lines below are the pipeline's own logging (from `load_projects()`), not notebook output — they just confirm the file loaded. The actual preview is in the next cell.

In [ ]:
raw_projects = load_projects(os.path.join(DATA_DIR, "projects.csv"))

In [ ]:
print("Shape (rows, columns):", raw_projects.shape)
raw_projects.head()

### Step 2 — What's missing before we touch anything

`budget` and `actual_cost` both have nulls — this is why they get filled with 0 *before* any derived column is calculated, not after.

In [ ]:
raw_projects[["budget", "actual_cost"]].isna().sum()

### Step 3 — Transform: fill nulls, add 6 derived columns

In [ ]:
projects_clean = transform_projects(raw_projects)
print(projects_clean.shape)
projects_clean[["project_id", "budget", "actual_cost", "budget_variance",
                "is_over_budget", "duration_days", "budget_utilisation_pct",
                "status_category", "risk_level"]].head(10)

### Step 4 — Sanity check: is `risk_level` skewed to one bucket?

In [ ]:
projects_clean["risk_level"].value_counts()

## Task 1.3 — `employees.csv`

### Step 1 — Load the raw file

Same as before: the `INFO` lines below are the pipeline's own logging (from `load_employees()`) — the null-count check is a quick sanity pass, not the real data-quality detection (that's `clean_employees()`, next).

In [15]:
raw_employees = load_employees(os.path.join(DATA_DIR, "employees.csv"))

2026-09-12 20:53:36,877 | INFO | Loading employees data from d:\presight\Full-Stack-Data-Engineer-Project\datasets\employees.csv
2026-09-12 20:53:36,883 | INFO | Loaded 1000 employee rows, 12 columns
2026-09-12 20:53:36,884 | INFO | Null counts per column:
email    10


In [ ]:
print("Shape (rows, columns):", raw_employees.shape)
raw_employees.head()

### Step 2 — Run the full cleaning pass

`clean_employees()` finds and fixes all 6 issue categories in one pass. The cells below re-derive each issue's condition (same logic as in `etl_pipeline.py`) just to show real before/after rows for each one.

In [16]:
employees_clean = clean_employees(raw_employees)
qs = employees_clean.attrs["quality_summary"]

summary = pd.DataFrame([
    ["1", "Blank email",                    qs["missing_email_fixed"],              "Placeholder address"],
    ["2", "hire_date not a valid date",      qs["invalid_hire_date_format_fixed"],   "Set to null"],
    ["3", "hire_date outside 1990–today",     qs["implausible_hire_date_fixed"],      "Set to null"],
    ["4", "years_experience outside 0–50",   qs["years_experience_out_of_range_fixed"], "Set to level median"],
    ["5", "Salary outside level's band",     qs["salary_level_mismatch_fixed"],      "Capped to level p95"],
    ["6", "Active under Inactive manager",   qs["status_conflicts_found"],           "Guardrail (0 found)"],
], columns=["Issue #", "Issue", "Rows found", "Fix applied"])

print(f"Total rows fixed: {sum(qs.values())} across {len(raw_employees)} employees")
summary

2026-09-12 20:53:48,703 | INFO | Cleaning employees data...
2026-09-12 20:53:48,704 | INFO | Issue 1 [Missing values] — blank emails: 10 rows
2026-09-12 20:53:48,706 | INFO | Issue 2 [Invalid date format] — non-date-shaped hire_date: 5 rows
2026-09-12 20:53:48,709 | INFO | Issue 3 [Implausible dates] — hire_date outside 1990-today: 3 rows
2026-09-12 20:53:48,712 | INFO | Issue 4 [Numeric out-of-range] — years_experience outside [0,50]: 5 rows
2026-09-12 20:53:48,720 | INFO | Issue 5 [Logical inconsistency] — salary inconsistent with level band: 3 rows
2026-09-12 20:53:48,723 | INFO | Issue 6 [Status conflict] — Active employee reporting to an Inactive manager: 0 rows
2026-09-12 20:53:48,724 | INFO | Data quality summary: {'missing_email_fixed': 10, 'invalid_hire_date_format_fixed': 5, 'implausible_hire_date_fixed': 3, 'years_experience_out_of_range_fixed': 5, 'salary_level_mismatch_fixed': 3, 'status_conflicts_found': 0}


Total rows fixed: 26 across 1000 employees


,Issue #,Issue,Rows found,Fix applied
0,1,Blank email,10,Placeholder address
1,2,hire_date not a valid date,5,Set to null
2,3,hire_date outside 1990–today,3,Set to null
3,4,years_experience outside 0–50,5,Set to level median
4,5,Salary outside level's band,3,Capped to level p95
5,6,Active under Inactive manager,0,Guardrail (0 found)


### Issue 1 — Blank emails

10 rows. Fixed with a placeholder address, not a guessed one.

In [17]:
missing_email = raw_employees["email"].isna() | (raw_employees["email"].astype(str).str.strip() == "")
print("rows affected:", missing_email.sum())

before = raw_employees.loc[missing_email, ["employee_id", "email"]].head()
after = employees_clean.loc[missing_email, ["employee_id", "email"]].head()
before.merge(after, on="employee_id", suffixes=("_before", "_after"))

rows affected: 10


,employee_id,email_before,email_after
0,EMP0358,NaN,unknown@presight.ai
1,EMP0373,NaN,unknown@presight.ai
2,EMP0461,NaN,unknown@presight.ai
3,EMP0536,NaN,unknown@presight.ai
4,EMP0647,NaN,unknown@presight.ai


### Issue 2 & 3 — Broken / implausible hire dates

8 rows total (5 non-date-shaped + 3 outside 1990–today). Set to null rather than guessed.

In [18]:
bad_hire_date = raw_employees["employee_id"].isin(
    employees_clean.loc[employees_clean["hire_date"].isna() & raw_employees["hire_date"].notna(), "employee_id"]
)
before = raw_employees.loc[bad_hire_date, ["employee_id", "hire_date"]]
after = employees_clean.loc[bad_hire_date, ["employee_id", "hire_date"]]
before.merge(after, on="employee_id", suffixes=("_before", "_after"))

,employee_id,hire_date_before,hire_date_after
0,EMP0038,-999,NaT
1,EMP0160,99999-01-01,NaT
2,EMP0175,-999,NaT
3,EMP0395,99999-01-01,NaT
4,EMP0551,99999-01-01,NaT
5,EMP0600,-999,NaT
6,EMP0855,-999,NaT
7,EMP0973,-999,NaT


### Issue 4 — `years_experience` out of range

5 rows outside [0, 50]. Replaced with the median for that employee's level.

In [19]:
yrs = pd.to_numeric(raw_employees["years_experience"], errors="coerce")
out_of_range = (yrs < 0) | (yrs > 50)
print("rows affected:", out_of_range.sum())

before = raw_employees.loc[out_of_range, ["employee_id", "level", "years_experience"]]
after = employees_clean.loc[out_of_range, ["employee_id", "level", "years_experience"]]
before.merge(after, on=["employee_id", "level"], suffixes=("_before", "_after"))

rows affected: 5


,employee_id,level,years_experience_before,years_experience_after
0,EMP0216,Junior,-1,1.0
1,EMP0356,Junior,-1,1.0
2,EMP0445,Senior,-1,9.0
3,EMP0583,Senior,-1,9.0
4,EMP0720,Mid,-1,4.0


In [ ]:
#full dataframe with the p95 and p05 values for each level

raw_employees.assign(
    p95_for_level=p95_by_level,
    p05_for_level=p05_by_level,
)

,employee_id,full_name,email,department,role,level,hire_date,salary,manager_id,region,status,years_experience,p95_for_level,p05_for_level
0,EMP0001,Anjali Desai,anjali.desai@presight.ai,Operations,Senior Data Engineer,Mid,2019-08-06,17839,EMP0000,Abu Dhabi,Active,6,23471.25,17396.0
1,EMP0002,Mohammed Mansour,mohammed.mansour@presight.ai,Operations,Senior Data Engineer,Senior,2014-08-20,33928,EMP0000,Dubai,Active,11,35485.30,25898.4
2,EMP0003,Adam Rahman,adam.rahman@presight.ai,Legal,Compliance Analyst,Mid,2021-02-17,18763,EMP0000,Abu Dhabi,Active,4,23471.25,17396.0
3,EMP0004,Fatima Al Suwaidi,fatima.al.suwaidi@presight.ai,Procurement,Procurement Analyst,Mid,2024-05-30,20100,EMP0000,Dubai,Active,2,23471.25,17396.0
4,EMP0005,Priya Chen,priya.chen@presight.ai,Procurement,Procurement Analyst,Junior,2024-02-21,14370,EMP0000,Abu Dhabi,Active,0,16778.40,12238.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,EMP0996,Mariam Al Khaja,mariam.al.khaja@presight.ai,IT,Network Engineer,Mid,2024-09-15,21985,EMP0028,Abu Dhabi,Active,3,23471.25,17396.0
996,EMP0997,Anjali Kapoor,anjali.kapoor2@presight.ai,Finance,Financial Analyst,Senior,2015-02-16,30594,EMP0038,Sharjah,Active,12,35485.30,25898.4
997,EMP0998,Carlos Johnson,carlos.johnson1@presight.ai,Data Science,BI Engineer,Junior,2024-04-21,12164,EMP0042,Abu Dhabi,Active,1,16778.40,12238.5
998,EMP0999,Min Kapoor,min.kapoor@presight.ai,Sales,Sales Analyst,Senior,2016-07-15,27174,EMP0047,Abu Dhabi,Active,9,35485.30,25898.4


### Issue 5 — Salary far outside the level's normal band

3 rows. Capped (winsorised) to the level's 95th percentile instead of dropped.

In [28]:
flagged = employees_clean["salary_flagged_outlier"]
print("rows affected:", flagged.sum())

before = raw_employees.loc[flagged, ["employee_id", "level", "salary"]]
after = employees_clean.loc[flagged, ["employee_id", "level", "salary"]]
before.merge(after, on=["employee_id", "level"], suffixes=("_before", "_after"))

rows affected: 3


,employee_id,level,salary_before,salary_after
0,EMP0356,Junior,76507,16778.4
1,EMP0867,Junior,66797,16778.4
2,EMP0900,Junior,71212,16778.4


### Issue 6 — Active employee under an Inactive manager

A guardrail check — finds 0 rows on this dataset, kept in anyway.

In [21]:
employees_clean.attrs["quality_summary"]["status_conflicts_found"]

0

## Recap

- `projects_clean` — 500 rows, 17 columns (11 raw + 6 derived)
- `employees_clean` — 1,000 rows, 13 columns (12 raw + 1 derived: `salary_flagged_outlier`)
- Every fix above was found by checking the *entire* column, not the first few rows — none of these 6 issues show up if you only look at the first 40 rows of `employees.csv`.